Modelling of df_modelling_full.xlsx

# Libraries

In [88]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [89]:
import pandas as pd
from pathlib import Path
import sys

from tqdm import tqdm

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [90]:
RANDOM_STATE = 42

In [91]:
# train/test split
sys.path.insert(1, '../utils_functionality/split_utils/')
from split_tools import get_train_test

In [92]:
# sklearn models pipeline
sys.path.insert(1, '../utils_functionality/models/')
from modelling2_utils import MLPipeline

# Splashing

## Train/test datasets creation

In [93]:
target = 'splashing'
train, test = get_train_test(
    dataset_filename='df_modelling_full',
    target=target)

## Numerical, categorical features selection

In [94]:
train.describe()

,splashing,wettability,roughness,liquid_density,surface_tension,viscosity,particle_mean_diameter,particle_density,droplet_diameter,inclination,roughness_binary,particle_liquid_density_ratio,volume_fraction_binary,particle_droplet_diameter_ratio,velocity,Re,We,We_Re
count,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000
mean,0.646465,0.912458,2.068956,1116.161616,0.067515,0.015695,0.000083,1102.356902,0.003145,9.242424,0.117845,0.990754,0.727273,0.025974,3.641049,3437.867206,759.440848,171.713124
std,0.478874,0.825530,3.405158,98.337949,0.008979,0.009989,0.000080,320.569332,0.000271,15.987172,0.322969,0.269354,0.446113,0.024387,1.056598,4881.530866,403.820050,84.738011
min,0.000000,0.000000,0.040000,820.000000,0.026900,0.001040,0.000041,450.000000,0.002130,0.000000,0.000000,0.381356,0.000000,0.011339,1.980571,256.976895,142.008866,52.684981
25%,0.000000,0.000000,0.040000,1000.000000,0.067900,0.002050,0.000041,1000.000000,0.002990,0.000000,0.000000,0.877193,0.000000,0.012652,3.961141,600.961715,629.826251,128.988793
50%,1.000000,1.000000,0.100000,1180.000000,0.067900,0.023100,0.000041,1000.000000,0.003210,0.000000,0.000000,1.000000,1.000000,0.013577,3.961141,672.716015,807.132068,152.487586
75%,1.000000,2.000000,2.490000,1180.000000,0.071300,0.023100,0.000041,1200.000000,0.003340,20.000000,0.000000,1.016949,1.000000,0.018694,3.961141,6185.563594,903.933380,236.927284
max,1.000000,2.000000,10.890000,1180.000000,0.073200,0.023100,0.000275,2200.000000,0.003660,45.000000,1.000000,1.864407,1.000000,0.103774,5.941712,19139.168057,2036.917752,472.780129


In [95]:
for column in train.columns:
    print(f'{column}:\t|\t{[round(x, 3) for x in train[column].unique()[:5]]}\t|\tUnique vales: {train[column].nunique()}')

splashing:	|	[0, 1]	|	Unique vales: 2
wettability:	|	[1, 2, 0]	|	Unique vales: 3
roughness:	|	[0.04, 2.49, 10.89, 0.1]	|	Unique vales: 4
liquid_density:	|	[1180, 1000, 820, 1060, 1140]	|	Unique vales: 5
surface_tension:	|	[0.068, 0.073, 0.027, 0.071, 0.069]	|	Unique vales: 5
viscosity:	|	[0.023, 0.001, 0.007, 0.002, 0.007]	|	Unique vales: 5
particle_mean_diameter:	|	[0.0, 0.0, 0.0]	|	Unique vales: 3
particle_density:	|	[1000, 2200, 1200, 450]	|	Unique vales: 4
droplet_diameter:	|	[0.003, 0.003, 0.003, 0.003, 0.003]	|	Unique vales: 105
inclination:	|	[45, 0, 20]	|	Unique vales: 3
roughness_binary:	|	[0, 1]	|	Unique vales: 2
particle_liquid_density_ratio:	|	[0.847, 1.0, 1.864, 1.017, 1.22]	|	Unique vales: 8
volume_fraction_binary:	|	[1, 0]	|	Unique vales: 2
particle_droplet_diameter_ratio:	|	[0.014, 0.044, 0.014, 0.013, 0.012]	|	Unique vales: 148
velocity:	|	[1.981, 3.961, 5.942]	|	Unique vales: 3
Re:	|	[300.481, 11864.38, 16568.235, 6246.415, 682.304]	|	Unique vales: 196
We:	|	[202.465,

In [96]:
binary_features = ['roughness_binary', 'volume_fraction_binary']
categorical_features = ['wettability']
numerical_features = ['droplet_diameter', 'particle_droplet_diameter_ratio',
                      'particle_liquid_density_ratio', 'Re', 'We', 'We_Re',
                      'roughness', 'liquid_density',
                        'surface_tension', 'viscosity', 'particle_mean_diameter',
                        'particle_density', 'inclination', 'velocity']

In [97]:
assert len((set(train.columns)) & set(binary_features + categorical_features + numerical_features)) + 1 \
    == len(train.columns), 'Some features are not selected'

## Modelling

Models:
* Sklearn models
    * LogReg
    * SVM
    * KNN
* CatBoost
* XGBoost

### Sklearn models

In [98]:
models = [
    LogisticRegression(),
    SVC(),
    KNeighborsClassifier()]
encoders = ['onehot', 'ordenc']

for model in tqdm(models):
    for encoder in encoders:
        pipeline = MLPipeline(train, test, target, model,
                 numerical_features, categorical_features,
                 random_state=RANDOM_STATE,
                 postfix=encoder)
        pipeline.full_pipeline()

  0%|          | 0/3 [00:00<?, ?it/s]C:\Users\dimaz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\linear_model\_logistic.py:444: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\dimaz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\linear_model\_logistic.py:444: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown 

logisticregression_splashing_onehot was saved in modelling2_models
logisticregression_splashing_ordenc was saved in modelling2_models
svc_splashing_onehot was saved in modelling2_models
svc_splashing_ordenc was saved in modelling2_models
kneighborsclassifier_splashing_onehot was saved in modelling2_models
kneighborsclassifier_splashing_ordenc was saved in modelling2_models


### Boosting Models

In [99]:
models =  [
    XGBClassifier(),
    CatBoostClassifier(verbose=False)
    ]

for model in tqdm(models):
    pipeline = MLPipeline(train, test, target, model,
                numerical_features, categorical_features,
                random_state=RANDOM_STATE,
                postfix='')
    pipeline.full_pipeline()

  0%|          | 0/2 [00:00<?, ?it/s]

 50%|█████     | 1/2 [00:00<00:00,  9.25it/s]C:\Users\dimaz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\catboost\core.py:1419: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  self._init_pool(data, label, cat_features, text_features, embedding_features, embedding_features_data, pairs, weight,


xgbclassifier_splashing was saved in modelling2_models


C:\Users\dimaz\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\catboost\core.py:1419: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  self._init_pool(data, label, cat_features, text_features, embedding_features, embedding_features_data, pairs, weight,
100%|██████████| 2/2 [00:12<00:00,  6.28s/it]

catboostclassifier_splashing was saved in modelling2_models
